# RAG 검색 품질 평가 PoC

LLM Wiki 기반 RAG 파이프라인 구축 및 Recall@k / MRR 측정

- **평가 대상**: `wiki/` 디렉토리 전체 문서 (50+ 파일)
- **평가 지표**: Recall@3, Recall@5, MRR
- **임베딩 모델**: `text-embedding-3-small` (OpenAI)
- **벡터 DB**: ChromaDB (로컬)
- **평가 쌍**: 20개 Q&A

## 1. 환경 설정

In [ ]:
import os
from pathlib import Path

# OpenAI API 키 설정
# OPENAI_API_KEY는 .env 또는 환경변수로 설정 (코드에 직접 넣지 않음)
from dotenv import load_dotenv
load_dotenv()

# 경로 설정
WIKI_DIR = Path(os.environ.get("WIKI_DIR", "./wiki"))
DB_DIR = Path("./chroma_db")

print(f"Wiki dir   : {WIKI_DIR}")
print(f"Wiki exists: {WIKI_DIR.exists()}")
print(f"API Key set: {bool(os.environ.get('OPENAI_API_KEY', '').startswith('sk-'))}")

## 2. Wiki 문서 로드

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

EXCLUDE_FILES = {"index.md", "log.md", "todo.md", "CLAUDE.md"}
EXCLUDE_DIRS = {"graphify-out"}

def load_wiki_docs(wiki_dir: Path):
    docs = []
    for md_file in wiki_dir.rglob("*.md"):
        if md_file.name in EXCLUDE_FILES:
            continue
        if any(d in md_file.parts for d in EXCLUDE_DIRS):
            continue
        try:
            loader = TextLoader(str(md_file), encoding="utf-8")
            loaded = loader.load()
            for doc in loaded:
                doc.metadata["source_file"] = md_file.stem
                doc.metadata["source_path"] = str(md_file.relative_to(wiki_dir))
            docs.extend(loaded)
        except Exception as e:
            print(f"[SKIP] {md_file.name}: {e}")
    return docs

raw_docs = load_wiki_docs(WIKI_DIR)
print(f"로드된 파일 수: {len(raw_docs)}")
for doc in raw_docs:
    print(f"  - {doc.metadata['source_path']}")

## 3. 텍스트 청킹

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

chunks = splitter.split_documents(raw_docs)
print(f"총 청크 수: {len(chunks)}")
print(f"\n--- 샘플 청크 ---")
print(f"source: {chunks[0].metadata['source_file']}")
print(chunks[0].page_content[:200])

## 4. 임베딩 & ChromaDB 생성

> 최초 1회만 실행. 이미 `chroma_db/` 폴더가 있으면 아래 셀 건너뛰고 '4-b. 기존 DB 로드'로 이동.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"]
)

# 신규 생성
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(DB_DIR)
)
print(f"ChromaDB 생성 완료: {len(chunks)}개 청크 저장됨")

In [ ]:
# 4-b. 기존 DB 재사용 (이미 생성된 경우)
# vectorstore = Chroma(
#     persist_directory=str(DB_DIR),
#     embedding_function=embeddings
# )
# print(f"기존 ChromaDB 로드 완료")

## 5. Q&A 평가 쌍 (20개)

각 질문이 어떤 wiki 페이지를 반환해야 하는지 정의한 ground truth.

In [ ]:
QA_PAIRS = [
    # Concepts
    {"question": "RAG 파이프라인의 주요 단계와 구조는?",               "expected": "concept-RAG"},
    {"question": "Ontology 기반 지식 표현 방식이란?",                   "expected": "concept-ontology"},
    {"question": "MLOps의 정의와 주요 구성요소는?",                     "expected": "concept-MLOps"},
    {"question": "MCP Model Context Protocol 개념과 활용",              "expected": "concept-MCP"},
    {"question": "클라우드 아키텍처의 주요 구성요소와 설계 원칙",        "expected": "concept-cloud-architecture"},
    {"question": "컨텍스트 엔지니어링이란 무엇이고 왜 필요한가?",        "expected": "concept-context-engineering"},
    {"question": "LLM 환각 발생 원인과 완화 방법",                      "expected": "concept-hallucination"},
    {"question": "AI 서브에이전트 패턴과 오케스트레이션",               "expected": "concept-subagents"},
    {"question": "SLO Service Level Objective 정의와 측정 방법",        "expected": "concept-SLO"},
    {"question": "Zettelkasten 방법론과 지식 연결 방식",                "expected": "concept-Zettelkasten"},
    # Entities
    {"question": "NAVER Cloud HyperCLOVA X AX 서비스",                 "expected": "entity-NAVER-Cloud"},
    {"question": "VGG-16 전이학습 킥보드 불법 주정차 분류 정확도",       "expected": "entity-VGG16-kickboard"},
    {"question": "GPUaaS PoC 프로젝트 구조와 목적",                     "expected": "entity-GPUaaS-PoC"},
    {"question": "NHN Cloud 서비스와 특징",                             "expected": "entity-NHN-Cloud"},
    {"question": "Anthropic Claude API 모델 종류와 특징",               "expected": "entity-Anthropic"},
    {"question": "AWS 클라우드 주요 서비스와 활용 사례",                 "expected": "entity-AWS"},
    # Summaries / Comparisons
    {"question": "RAG와 Mem0의 차이점과 각 장단점",                     "expected": "comparison-RAG-vs-Mem0"},
    {"question": "샘 알트만이 언급한 RAG 관련 발언",                    "expected": "summary-RAG-샘호트만-2026"},
    {"question": "LLM Wiki 지식 관리 시스템 설계 원칙",                 "expected": "concept-LLM-Wiki"},
    {"question": "하네스 엔지니어링 개념과 AI 시스템 적용",             "expected": "concept-harness-engineering"},
]

print(f"평가 쌍 수: {len(QA_PAIRS)}개")

## 6. 평가 함수 정의

In [ ]:
def recall_at_k(vectorstore, question: str, expected_source: str, k: int) -> int:
    """정답 문서가 top-k 결과 안에 있으면 1, 없으면 0"""
    results = vectorstore.similarity_search(question, k=k)
    retrieved = [r.metadata.get("source_file", "") for r in results]
    return 1 if expected_source in retrieved else 0

def reciprocal_rank(vectorstore, question: str, expected_source: str, k: int = 10) -> float:
    """정답 문서의 순위의 역수. 정답 없으면 0"""
    results = vectorstore.similarity_search(question, k=k)
    for rank, result in enumerate(results, start=1):
        if result.metadata.get("source_file", "") == expected_source:
            return 1.0 / rank
    return 0.0

print("평가 함수 준비 완료")

## 7. 평가 실행

In [ ]:
import pandas as pd

rows = []
for i, pair in enumerate(QA_PAIRS, 1):
    q   = pair["question"]
    exp = pair["expected"]
    r3  = recall_at_k(vectorstore, q, exp, k=3)
    r5  = recall_at_k(vectorstore, q, exp, k=5)
    rr  = reciprocal_rank(vectorstore, q, exp, k=10)
    rows.append({
        "#": i,
        "question": q[:35] + "..." if len(q) > 35 else q,
        "expected": exp,
        "Recall@3": r3,
        "Recall@5": r5,
        "RR": round(rr, 3)
    })
    print(f"[{i:02d}] R@3={r3} R@5={r5} RR={rr:.3f} | {exp}")

df = pd.DataFrame(rows)
df

## 8. 결과 요약

In [ ]:
recall3 = df["Recall@3"].mean()
recall5 = df["Recall@5"].mean()
mrr     = df["RR"].mean()

print("=" * 40)
print(f"  Recall@3 : {recall3:.3f}  ({recall3*100:.1f}%)")
print(f"  Recall@5 : {recall5:.3f}  ({recall5*100:.1f}%)")
print(f"  MRR      : {mrr:.3f}")
print("=" * 40)

# 실패 케이스 출력
failed = df[df["Recall@5"] == 0]
if len(failed) > 0:
    print(f"\nRecall@5 실패 ({len(failed)}건):")
    for _, row in failed.iterrows():
        print(f"  - {row['expected']}: {row['question']}")

# CSV 저장
Path("results").mkdir(exist_ok=True)
df.to_csv("results/eval_results.csv", index=False, encoding="utf-8-sig")
print("\nresults/eval_results.csv 저장 완료")

---

## 9. 비교 실험 — 구조화 vs 비구조화

**가설**: 개념·엔티티별로 분리된 구조화 문서가 하나로 합친 비구조화 문서보다 RAG 검색 품질이 높다.

| 조건 | 문서 형태 | 예상 결과 |
|------|----------|----------|
| **구조화 (현재)** | 50개 파일, 파일당 하나의 개념 | Recall 높음 |
| **비구조화 (baseline)** | 전체 내용을 1개 파일로 합침 | Recall 낮음 |

평가 방식 차이:
- 구조화: `source_file` 메타데이터 정확 매칭
- 비구조화: 검색된 청크에 핵심 키워드 포함 여부 (soft match)

In [ ]:
from langchain_core.documents import Document

# 평가 쌍별 핵심 키워드 정의 (비구조화 soft match용)
QA_KEYWORDS = {
    "concept-RAG":               ["RAG", "검색 증강", "벡터 검색", "임베딩"],
    "concept-ontology":          ["Ontology", "온톨로지", "OWL", "지식 그래프"],
    "concept-MLOps":             ["MLOps", "ML 파이프라인", "모델 배포"],
    "concept-MCP":               ["MCP", "Model Context Protocol"],
    "concept-cloud-architecture":["클라우드 아키텍처", "IaaS", "SaaS", "cloud"],
    "concept-context-engineering":["컨텍스트 엔지니어링", "context engineering"],
    "concept-hallucination":     ["환각", "hallucination"],
    "concept-subagents":         ["서브에이전트", "subagent", "오케스트레이션"],
    "concept-SLO":               ["SLO", "Service Level Objective"],
    "concept-Zettelkasten":      ["Zettelkasten", "제텔카스텐"],
    "entity-NAVER-Cloud":        ["NAVER Cloud", "HyperCLOVA", "네이버클라우드"],
    "entity-VGG16-kickboard":    ["VGG-16", "VGG16", "킥보드", "81.70"],
    "entity-GPUaaS-PoC":         ["GPUaaS", "GPU as a Service"],
    "entity-NHN-Cloud":          ["NHN Cloud", "NHN클라우드"],
    "entity-Anthropic":          ["Anthropic", "Claude"],
    "entity-AWS":                ["AWS", "Amazon Web Services"],
    "comparison-RAG-vs-Mem0":    ["Mem0", "RAG", "비교"],
    "summary-RAG-샘호트만-2026": ["샘 알트만", "Sam Altman", "호트만"],
    "concept-LLM-Wiki":          ["LLM Wiki", "지식 관리", "wiki"],
    "concept-harness-engineering":["하네스", "harness engineering"],
}

# 비구조화 문서 생성: 전체 wiki 내용을 1개 파일로 합침
def load_merged_doc(wiki_dir: Path):
    all_text = ""
    for md_file in sorted(wiki_dir.rglob("*.md")):
        if md_file.name in EXCLUDE_FILES:
            continue
        if any(d in md_file.parts for d in EXCLUDE_DIRS):
            continue
        try:
            all_text += md_file.read_text(encoding="utf-8") + "\n\n"
        except:
            pass
    return [Document(page_content=all_text, metadata={"source_file": "merged"})]

merged_raw = load_merged_doc(WIKI_DIR)
merged_chunks = splitter.split_documents(merged_raw)

print(f"구조화   : {len(chunks)}개 청크 (파일 {len(raw_docs)}개)")
print(f"비구조화 : {len(merged_chunks)}개 청크 (파일 1개)")
print(f"청크당 평균 길이 — 구조화: {sum(len(c.page_content) for c in chunks)//len(chunks)}자")
print(f"청크당 평균 길이 — 비구조화: {sum(len(c.page_content) for c in merged_chunks)//len(merged_chunks)}자")

In [ ]:
# 비구조화 벡터 DB 생성
vs_unstructured = Chroma.from_documents(
    documents=merged_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db_unstructured"
)
print("비구조화 ChromaDB 생성 완료")

In [ ]:
# 비구조화 평가: 키워드 기반 soft match
def recall_at_k_keyword(vs, question, expected_source, k):
    keywords = QA_KEYWORDS.get(expected_source, [])
    results = vs.similarity_search(question, k=k)
    for result in results:
        content = result.page_content
        if any(kw in content for kw in keywords):
            return 1
    return 0

def reciprocal_rank_keyword(vs, question, expected_source, k=10):
    keywords = QA_KEYWORDS.get(expected_source, [])
    results = vs.similarity_search(question, k=k)
    for rank, result in enumerate(results, start=1):
        if any(kw in result.page_content for kw in keywords):
            return 1.0 / rank
    return 0.0

# 비교 실행
comparison_rows = []
for i, pair in enumerate(QA_PAIRS, 1):
    q, exp = pair["question"], pair["expected"]

    # 구조화 (exact match — 이미 계산된 값 재사용)
    s_r3 = recall_at_k(vectorstore, q, exp, k=3)
    s_r5 = recall_at_k(vectorstore, q, exp, k=5)
    s_rr = reciprocal_rank(vectorstore, q, exp, k=10)

    # 비구조화 (keyword soft match)
    u_r3 = recall_at_k_keyword(vs_unstructured, q, exp, k=3)
    u_r5 = recall_at_k_keyword(vs_unstructured, q, exp, k=5)
    u_rr = reciprocal_rank_keyword(vs_unstructured, q, exp, k=10)

    comparison_rows.append({
        "#": i, "expected": exp,
        "구조화 R@3": s_r3, "구조화 R@5": s_r5, "구조화 RR": round(s_rr, 3),
        "비구조화 R@3": u_r3, "비구조화 R@5": u_r5, "비구조화 RR": round(u_rr, 3),
    })
    print(f"[{i:02d}] 구조화 R@3={s_r3} RR={s_rr:.2f} | 비구조화 R@3={u_r3} RR={u_rr:.2f} | {exp}")

df_cmp = pd.DataFrame(comparison_rows)
df_cmp

In [ ]:
# 최종 비교 요약
print("=" * 55)
print(f"{'지표':<12} {'구조화':>10} {'비구조화':>10} {'차이(↑)':>10}")
print("-" * 55)

for metric, s_col, u_col in [
    ("Recall@3", "구조화 R@3", "비구조화 R@3"),
    ("Recall@5", "구조화 R@5", "비구조화 R@5"),
    ("MRR",      "구조화 RR",  "비구조화 RR"),
]:
    s = df_cmp[s_col].mean()
    u = df_cmp[u_col].mean()
    diff = s - u
    print(f"{metric:<12} {s:>9.3f}  {u:>9.3f}  {diff:>+9.3f}")

print("=" * 55)

# 결과 저장
df_cmp.to_csv("results/comparison_results.csv", index=False, encoding="utf-8-sig")
print("\nresults/comparison_results.csv 저장 완료")